# 02 · 초기 피처 설계와 입력 검증

**목표:** DAY 1 EDA의 관찰을 모델 입력 후보로 옮기고, 미래 정보 누수와 분할 중복을 점검한다. 학습·후보 선택·최종 평가는 [03_modeling.ipynb](03_modeling.ipynb)에서 수행한다. 이 노트북은 저장된 가공 데이터를 사용하며 원본 MAT가 없어도 실행된다.

## 1. 사전에 고정한 셀과 분할

한 셀이 한 행이다. 완료되지 않았을 가능성이 있는 수명 라벨은 제외하고, Batch 1의 충전 정책 그룹을 기준으로 Train/CV와 Hold-out을 분리했다. 외부 배치의 라벨은 피처 및 모델 선택에 사용하지 않는다.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
assert (ROOT / "src/preprocess.py").is_file()
sys.path.insert(0, str(ROOT / "src"))
from preprocess import load_data
from features import FEATURE_SETS, feature_matrix
train, holdout, external, plan = load_data(ROOT)
cohorts = {"Train/CV": train, "Hold-out": holdout, **external}
display(pd.DataFrame([{"집단": k, "셀 수": len(v), "정책 그룹": v.policy_group.nunique()} for k,v in cohorts.items()]))
print("Train/Hold-out 정책 그룹 겹침:", len(set(train.policy_group) & set(holdout.policy_group)))

Matplotlib is building the font cache; this may take a moment.


,집단,셀 수,정책 그룹
0,Train/CV,29,16
1,Hold-out,7,4
2,Batch 2,39,9
3,Batch 3,44,8


Train/Hold-out 정책 그룹 겹침: 0


## 2. EDA 관찰 → 후보 입력

- **G0 ΔQ:** 10~100사이클의 방전곡선 변화량 분산. DAY 1에서 수명과 강한 음의 상관을 보여 기본형으로 둔다.
- **G1 열화:** 초기 방전용량의 10~100사이클 기울기를 추가한다.
- **G2 충전:** 평균 충전시간과 1단계 C-rate를 더한다.
- **G3 온도:** 온도 신호를 추가하되 다른 온도 변수와의 중복을 피한다.
- **G4 전류:** 공통 cycle 2~5의 시간 가중 RMS 전류를 G0에 별도로 더한다.

피처 추가가 도움이 되는지는 Train/CV에서만 판단한다. 전체 수명으로 계산한 knee, 종료 QD, 수명 라벨, 셀 ID는 입력에서 제외한다.

In [2]:
audit = []
for group, names in FEATURE_SETS.items():
    x = feature_matrix(train, names)
    audit.append({"피처 묶음": group, "입력 변수": ", ".join(names), "변수 수": len(names), "Train 결측값": int(x.isna().sum().sum())})
display(pd.DataFrame(audit))
assert all(set(names) <= set(plan["input_whitelist"]) for names in FEATURE_SETS.values())
assert all("cycle_life" not in names and "knee_cycle" not in names for names in FEATURE_SETS.values())

,피처 묶음,입력 변수,변수 수,Train 결측값
0,G0_delta,log_var_delta_q,1,0
1,G1_trend,"log_var_delta_q, qd_slope_10_100",2,0
2,G2_charge,"log_var_delta_q, qd_slope_10_100, mean_charget...",4,0
3,G3_thermal,"log_var_delta_q, qd_slope_10_100, mean_charget...",5,0
4,G4_current,"log_var_delta_q, charge_rms_a",2,0


## 3. 핵심 입력의 관측 범위와 중복

다음 표는 입력값의 범위를 확인하는 진단 자료다. Batch 2·3의 라벨을 이용한 피처 선정에는 쓰지 않는다. 상관은 작은 표본과 공통 충전정책의 영향을 받을 수 있으므로 인과 효과로 해석하지 않는다.

In [3]:
names = list(dict.fromkeys(sum(FEATURE_SETS.values(), [])))
summary = pd.DataFrame({
    "Train 중앙값": feature_matrix(train, names).median(),
    "Train 최솟값": feature_matrix(train, names).min(),
    "Train 최댓값": feature_matrix(train, names).max(),
    "Batch 2 결측": feature_matrix(external["Batch 2"], names).isna().sum(),
    "Batch 3 결측": feature_matrix(external["Batch 3"], names).isna().sum(),
})
display(summary.round(3))
display(feature_matrix(train, names).corr(method="spearman").round(2))

,Train 중앙값,Train 최솟값,Train 최댓값,Batch 2 결측,Batch 3 결측
log_var_delta_q,-3.777,-4.146,-3.350,0,0
qd_slope_10_100,-0.000,-0.000,0.000,0,0
mean_chargetime,10.801,8.987,14.928,0,0
c1,6.000,5.400,8.000,0,0
mean_tavg,33.077,30.375,33.895,0,0
charge_rms_a,3.138,2.934,3.426,0,0


,log_var_delta_q,qd_slope_10_100,mean_chargetime,c1,mean_tavg,charge_rms_a
log_var_delta_q,1.00,-0.73,-0.64,0.38,0.08,0.71
qd_slope_10_100,-0.73,1.00,0.64,-0.20,0.02,-0.41
mean_chargetime,-0.64,0.64,1.00,0.16,0.23,-0.60
c1,0.38,-0.20,0.16,1.00,0.29,0.16
mean_tavg,0.08,0.02,0.23,0.29,1.00,0.02
charge_rms_a,0.71,-0.41,-0.60,0.16,0.02,1.00


## 4. 모델로 넘기는 규칙

`src/features.py`의 화이트리스트 함수로만 입력을 만들고, 결측값 대체·표준화는 `src/train.py`의 각 학습 fold 내부에서 적합한다. G0~G4와 후보 모델의 성능은 [03_modeling.ipynb](03_modeling.ipynb), 요약 성능은 [model_performance.csv](../results/model_performance.csv)에서 확인한다. 결측이 없는 현재 데이터라도 파이프라인의 적합 범위를 유지해 재현 시 누수를 막는다.